In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [4]:
data=pd.read_csv("final.csv")

In [5]:
data.head()

,Unnamed: 0,Category_ID,category_name,Product_ID,Product_Name,Launch_Date,Price,sale_id,sale_date,store_id,quantity
0,0,CAT-4,Smartphone,P-38,iPhone 13 Pro,2021-03-22,308,YG-8782,16-06-2023,ST-10,10
1,1,CAT-5,Wearable,P-48,Apple Watch Nike Edition,2020-06-24,884,QX-999001,13-04-2022,ST-63,10
2,2,CAT-10,Accessories,P-79,Magic Trackpad,2024-05-25,1242,JG-46890,05-07-2021,ST-26,5
3,3,CAT-3,Tablet,P-24,iPad mini (6th Generation),2022-11-27,573,XJ-1731,20-07-2022,ST-15,9
4,4,CAT-8,Subscription Service,P-69,Apple TV+,2024-11-04,404,FG-95080,18-03-2022,ST-35,7


In [6]:
data=data.drop("Unnamed: 0",axis=1)

In [7]:
data.isna().sum()

Category_ID      0
category_name    0
Product_ID       0
Product_Name     0
Launch_Date      0
Price            0
sale_id          0
sale_date        0
store_id         0
quantity         0
dtype: int64

In [8]:
data.head()

,Category_ID,category_name,Product_ID,Product_Name,Launch_Date,Price,sale_id,sale_date,store_id,quantity
0,CAT-4,Smartphone,P-38,iPhone 13 Pro,2021-03-22,308,YG-8782,16-06-2023,ST-10,10
1,CAT-5,Wearable,P-48,Apple Watch Nike Edition,2020-06-24,884,QX-999001,13-04-2022,ST-63,10
2,CAT-10,Accessories,P-79,Magic Trackpad,2024-05-25,1242,JG-46890,05-07-2021,ST-26,5
3,CAT-3,Tablet,P-24,iPad mini (6th Generation),2022-11-27,573,XJ-1731,20-07-2022,ST-15,9
4,CAT-8,Subscription Service,P-69,Apple TV+,2024-11-04,404,FG-95080,18-03-2022,ST-35,7


In [9]:
data["quantity_1"] = pd.cut(data["quantity"], bins=[0,2, 4, 8, 10, np.inf], labels=[1, 2, 3, 4,5])
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(data, data['quantity_1']):
    strat_train_set = data.loc[train_index]  
    strat_test_set = data.loc[test_index]    


In [10]:
data_cat = strat_train_set[["Product_ID", "Product_Name", "store_id", "sale_id", "category_name", "Category_ID"]]
data_no = strat_train_set.drop(["Product_ID", "Product_Name", "store_id", "sale_id", "category_name", "Category_ID"], axis=1)

In [11]:
data_no.head()

,Launch_Date,Price,sale_date,quantity,quantity_1
263519,2021-01-17,1351,29-03-2024,7,3
652400,2023-05-07,1761,06-05-2022,5,3
919743,2022-07-12,1631,19-09-2020,4,2
206013,2024-11-04,404,02-05-2020,7,3
985981,2020-09-27,555,30-01-2022,10,4


In [12]:
data_no = data_no.drop(["Launch_Date", "sale_date"], axis=1,errors="ignore")
data_num_only = data_no.select_dtypes(include=["number"])

mypipeline = Pipeline([ 
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

scaled_array = mypipeline.fit_transform(data_num_only)
scaled_data = pd.DataFrame(scaled_array, columns=data_num_only.columns, index=data_num_only.index)
scaled_data

,Price,quantity
263519,0.548246,0.521754
652400,1.370701,-0.174018
919743,1.109923,-0.521904
206013,-1.351423,0.521754
985981,-1.048519,1.565412
...,...,...
845120,1.747827,0.521754
372220,-1.243100,-1.217675
978422,1.779922,0.521754
522472,0.676629,-0.521904


In [13]:
proper_data=strat_train_set[["Launch_Date", "sale_date", "Product_Name"]]
proper_data_1 = pd.merge(proper_data, scaled_data, left_index=True, right_index=True, how="outer")
proper_data_1.head()

,Launch_Date,sale_date,Product_Name,Price,quantity
0,2021-03-22,16-06-2023,iPhone 13 Pro,-1.543998,1.565412
1,2020-06-24,13-04-2022,Apple Watch Nike Edition,-0.388550,1.565412
2,2024-05-25,05-07-2021,Magic Trackpad,0.329594,-0.174018
3,2022-11-27,20-07-2022,iPad mini (6th Generation),-1.012412,1.217526
4,2024-11-04,18-03-2022,Apple TV+,-1.351423,0.521754


In [17]:

from sklearn.tree import DecisionTreeRegressor

proper_data_1["Launch_Date"] = pd.to_datetime(proper_data_1["Launch_Date"])
proper_data_1["sale_date"] = pd.to_datetime(proper_data_1["sale_date"], dayfirst=True)

proper_data_1["Launch_Year"] = proper_data_1["Launch_Date"].dt.year
proper_data_1["Launch_Month"] = proper_data_1["Launch_Date"].dt.month
proper_data_1["Launch_Day"] = proper_data_1["Launch_Date"].dt.day

proper_data_1["Sale_Year"] = proper_data_1["sale_date"].dt.year
proper_data_1["Sale_Month"] = proper_data_1["sale_date"].dt.month
proper_data_1["Sale_Day"] = proper_data_1["sale_date"].dt.day

X = proper_data_1.drop(columns=["Price", "Launch_Date", "sale_date","Product_Name"])
Y = proper_data_1["Price"]

X_encoded = pd.get_dummies(X)

model = DecisionTreeRegressor(random_state=40)
model.fit(X_encoded, Y)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,40
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [18]:
# data test
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, Y_train, Y_test = train_test_split(
    X_encoded,
    Y,
    test_size=0.2,
    random_state=40
)

model = DecisionTreeRegressor(random_state=40)

model.fit(X_train, Y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(Y_test, predictions)

mse = mean_squared_error(Y_test, predictions)

r2 = r2_score(Y_test, predictions)

print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("R2 Score:", r2)

Mean Absolute Error: 0.0029358687989355435
Mean Squared Error: 0.0006873486171946112
R2 Score: 0.9993137102268674


In [27]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
predictions_actual = price_scaler.inverse_transform(
    predictions.reshape(-1, 1)
).ravel()

Y_test_actual = price_scaler.inverse_transform(
    Y_test.to_numpy().reshape(-1, 1)
).ravel()
mae = mean_absolute_error(
    Y_test_actual,
    predictions_actual
)

mse = mean_squared_error(
    Y_test_actual,
    predictions_actual
)

rmse = np.sqrt(mse)

r2 = r2_score(
    Y_test_actual,
    predictions_actual
)

print("=" * 50)
print("MODEL EVALUATION REPORT")
print("=" * 50)

print(f"Actual MAE  : {mae:.2f}")
print(f"Actual MSE  : {mse:.2f}")
print(f"RMSE        : {rmse:.2f}")
print(f"R2 Score    : {r2:.4f}")

print("=" * 50)

results = pd.DataFrame({
    "Actual_Price": Y_test_actual,
    "Predicted_Price": predictions_actual,
    "Error": np.abs(Y_test_actual - predictions_actual)
})

print("\nSample Predictions:")
print(results.head(20))
print("\nError Statistics:")
print(results["Error"].describe())

MODEL EVALUATION REPORT
Actual MAE  : 1.46
Actual MSE  : 170.91
RMSE        : 13.07
R2 Score    : 0.9993

Sample Predictions:
    Actual_Price  Predicted_Price         Error
0    1761.289232      1761.289232  2.114575e-11
1     680.977844       680.977844  2.273737e-12
2    1476.207060      1476.207060  1.114131e-11
3    1842.312586      1842.312586  1.296030e-10
4    1839.311721      1839.311721  6.366463e-11
5     367.887599       367.887599  1.084572e-10
6    1434.194951      1434.194951  2.091838e-11
7     448.910954       448.910954  3.228706e-11
8    1744.284331      1744.284331  4.956746e-11
9    1744.284331      1744.284331  4.956746e-11
10    571.946417       571.946417  6.821210e-12
11    923.047618       923.047618  7.958079e-13
12   1259.144494      1259.144494  2.432898e-11
13   1826.307973      1826.307973  1.705303e-11
14    437.907782       437.907782  1.103899e-10
15   1826.307973      1826.307973  1.409717e-11
16    503.926811       503.926811  2.899014e-11
17   1415.

In [28]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(
    random_state=40,
    max_depth=20,
    min_samples_split=10
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=40
)

scores = cross_val_score(
    model,
    X_encoded,
    Y,
    cv=kf,
    scoring="r2"
)

print("=" * 50)
print("CROSS VALIDATION RESULTS")
print("=" * 50)

print("R2 Scores for each fold:")
print(scores)

print("\nAverage R2 Score:", scores.mean())
print("Standard Deviation:", scores.std())

CROSS VALIDATION RESULTS
R2 Scores for each fold:
[0.99957199 0.99956027 0.99957659 0.99958904 0.99957511]

Average R2 Score: 0.9995745983139483
Standard Deviation: 9.21780147117501e-06
